# Color Diversity
## Notebook init
### Libs

In [ ]:
from pathlib import Path

import torch
from clearml import Dataset, Logger, Task, TaskTypes
from torch.utils.data import ConcatDataset, DataLoader, SequentialSampler

### Chromatica modules

In [ ]:
from chromatica.charts import color_diversity
from chromatica.datasets.dataset import ImageDataset

### Check for CUDA or MPS

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS is used")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is used")
else:
    device = torch.device("cpu")
    print("CPU is used")

### ClearML init

In [ ]:
task = Task.init(
    project_name="Chromatica",
    task_name="Analyze diversity of colors in datasets",
    task_type=TaskTypes.data_processing,
)

In [ ]:
params = {
    "batch_size": 32,
    "num_workers": 6,
    "seed": 42,
}
params = task.connect_configuration(params)

## `Food101`
### Prepare dataset `Food101`

In [ ]:
path = Path(
    Dataset.get(dataset_project="Colorization", dataset_name="Food101").get_local_copy()
)

In [ ]:
dataset1 = ImageDataset(path / "train")
dataset2 = ImageDataset(path / "test")

dataset = ConcatDataset([dataset1, dataset2])

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=params["batch_size"],
    num_workers=params["num_workers"],
    persistent_workers=True,
    shuffle=False,
    sampler=SequentialSampler(dataset),
    pin_memory=(device == torch.device("cuda")),
)

### Color Diversity (`Food101`)
#### Color distribution histogram

In [ ]:
%%time

hist = color_diversity.compute_histogram(
    loader,
    nbins=250,
    sample_fraction=0.5,
    rng=torch.Generator().manual_seed(params["seed"]),
)

In [ ]:
fig, _ = color_diversity.plot_histogram(hist, title="Food101 : Hue histogram")

Logger.current_logger().report_matplotlib_figure(
    title="Hue histogram",
    series="dataset=Food101",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=True,
)

#### Top Colors Pie Chart (Modes)

In [ ]:
%%time

top16_modes = color_diversity.compute_top_colors_modes(
    loader,
    topk=16,
    sample_fraction=0.5,
    rng=torch.Generator().manual_seed(params["seed"]),
    input_scale=110.0,
    bin_size=5.0,
    merge_radius=5.0,
)

In [ ]:
fig, _ = color_diversity.plot_top_colors_pie(
    top16_modes,
    l_value=50.0,
    title="Food101 : Top-16 Pie (Modes)",
    include_other=False,
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Pie (Modes)",
    series="dataset=Food101",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

#### Top Colors Palette Chart (Modes)

In [ ]:
fig, _ = color_diversity.plot_top_colors_palette(
    top16_modes,
    l_value=50.0,
    title="Food101 : Top-16 Palette (Modes)",
    cols=8,
    tile_px=72,
    spacing_px=2,
    fmt="{:.1f}%",
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Palette (Modes)",
    series="dataset=Food101",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

### Top Colors Pie Chart (K-Means)

In [ ]:
top16_km = color_diversity.compute_top_colors_kmeans(
    loader,
    topk=16,
    input_scale=128.0,
    chroma_floor=8.0,
    sample_fraction=0.05,
    rng=torch.Generator().manual_seed(params["seed"]),
    init_centers_ab=top16_modes.centers_ab,
)

In [ ]:
fig, _ = color_diversity.plot_top_colors_pie(
    top16_km,
    l_value=50.0,
    title="Food101 : Top-16 Pie (K-Means)",
    include_other=False,
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Pie (K-Means)",
    series="dataset=Food101",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

### Top Colors Palette (K-Means)

In [ ]:
fig, _ = color_diversity.plot_top_colors_palette(
    top16_km,
    l_value=50.0,
    title="Food101 : Top-16 Palette (K-Means)",
    cols=8,
    tile_px=72,
    spacing_px=2,
    fmt="{:.1f}%",
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Palette (K-Means)",
    series="dataset=Food101",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

## `COCO`
### Prepare dataset `COCO`

In [ ]:
path = Path(
    Dataset.get(dataset_project="Colorization", dataset_name="COCO").get_local_copy()
)

In [ ]:
dataset1 = ImageDataset(path / "train")
dataset2 = ImageDataset(path / "test")

dataset = ConcatDataset([dataset1, dataset2])

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=params["batch_size"],
    num_workers=params["num_workers"],
    persistent_workers=True,
    shuffle=False,
    sampler=SequentialSampler(dataset),
    pin_memory=(device == torch.device("cuda")),
)

### Color Diversity (`COCO`)
#### Color distribution histogram

In [ ]:
%%time

hist = color_diversity.compute_histogram(
    loader,
    nbins=250,
    sample_fraction=0.5,
    rng=torch.Generator().manual_seed(params["seed"]),
)

In [ ]:
fig, _ = color_diversity.plot_histogram(hist, title="COCO : Hue histogram")

Logger.current_logger().report_matplotlib_figure(
    title="COCO : Hue histogram",
    series="dataset=Food101",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

#### Top Colors Pie Chart (Modes)

In [ ]:
%%time

top16_modes = color_diversity.compute_top_colors_modes(
    loader,
    topk=16,
    sample_fraction=0.5,
    rng=torch.Generator().manual_seed(params["seed"]),
    input_scale=110.0,
    bin_size=5.0,
    merge_radius=5.0,
)

In [ ]:
fig, _ = color_diversity.plot_top_colors_pie(
    top16_modes,
    l_value=50.0,
    title="COCO : Top-16 Pie (Modes)",
    include_other=False,
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Pie (Modes)",
    series="dataset=COCO",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

#### Top Colors Palette Chart (Modes)

In [ ]:
fig, _ = color_diversity.plot_top_colors_palette(
    top16_modes,
    l_value=50.0,
    title="COCO : Top-16 Palette (Modes)",
    cols=8,
    tile_px=72,
    spacing_px=2,
    fmt="{:.1f}%",
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Palette (Modes)",
    series="dataset=COCO",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

### Top Colors Pie Chart (K-Means)

In [ ]:
top16_km = color_diversity.compute_top_colors_kmeans(
    loader,
    topk=16,
    input_scale=128.0,
    chroma_floor=8.0,
    sample_fraction=0.05,
    rng=torch.Generator().manual_seed(params["seed"]),
    init_centers_ab=top16_modes.centers_ab,
)

In [ ]:
fig, _ = color_diversity.plot_top_colors_pie(
    top16_km,
    l_value=50.0,
    title="COCO : Top-16 Pie (K-Means)",
    include_other=False,
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Pie (K-Means)",
    series="dataset=COCO",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

### Top Colors Palette (K-Means)

In [ ]:
fig, _ = color_diversity.plot_top_colors_palette(
    top16_km,
    l_value=50.0,
    title="COCO : Top-16 Palette (K-Means)",
    cols=8,
    tile_px=72,
    spacing_px=2,
    fmt="{:.1f}%",
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Palette (K-Means)",
    series="dataset=COCO",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

In [ ]:
task.mark_completed()